# Autenticação de Saída (Outbound Auth)

Outbound Auth permite que agentes e o AgentCore Gateway acessem com segurança recursos AWS e serviços de terceiros em nome de usuários que foram autenticados e autorizados durante Inbound Auth. Para integrar autorização com um recurso AWS ou serviço de terceiros, é necessário configurar tanto Inbound Auth quanto Outbound Auth.

Com acesso suficiente e delegação de permissão segura suportada pelo AgentCore Identity, agentes podem acessar perfeitamente e com segurança recursos AWS e ferramentas de terceiros como GitHub, Google, Salesforce e Slack. Agentes podem realizar ações nesses serviços em nome de usuários ou independentemente, desde que haja consentimento pré-autorizado do usuário. Adicionalmente, você pode reduzir fadiga de consentimento usando um cofre de tokens seguro e criar experiências simplificadas de agentes de IA.

## Configuração de Autenticação de Saída

Primeiro, você registra sua aplicação cliente com provedores de terceiros e então cria um Outbound Auth. Você especifica como deseja validar acesso ao recurso AWS ou serviço de terceiros ou alvos AgentCore Gateway. Você pode usar OAuth 2LO/3LO ou chaves API. Com OAuth, você pode selecionar de provedores que AgentCore Identity fornece. Nesse caso, você insere os detalhes de configuração para os provedores do AgentCore Identity. Alternativamente, você pode fornecer detalhes para um provedor personalizado.

Quando um usuário quer acesso a um recurso AWS ou serviço de terceiros ou alvo AgentCore Gateway, o Outbound Auth confirma que os tokens de acesso fornecidos pelo Incoming Auth são válidos e, se for o caso, permite acesso ao recurso.

<div style="text-align:center">
    <img src="images/outbound_auth.png" width="90%"/>
</div>


Aqui estão os vários parâmetros que você pode usar com o decorador @require_access_token.


| Nome do Parâmetro    | Descrição                                                                |
|:---------------------|:-------------------------------------------------------------------------|
| provider_name        | O nome do provedor de credenciais                                        |
| into                 | Nome do parâmetro para injetar o token                                   |
| scopes               | Escopos OAuth2 a serem solicitados                                       |
| on_auth_url	       | Callback para manipular URLs de autorização                              |
| auth_flow            | Tipo de fluxo de autenticação ("M2M" ou "USER_FEDERATION")               |
| callback_url         | URL de callback OAuth2                                                   |
| force_authentication | Forçar re-autenticação                                                   |
| token_poller         | Implementação personalizada do token poller                              |

		


# Hospedando Strands Agents no Amazon Bedrock AgentCore Runtime

## Visão Geral


Neste tutorial, desenvolveremos um agente de agendamento usando Strands agents que pode listar os eventos do Google Calendar do usuário. Configuraremos um provedor de credenciais para ajudar com o gerenciamento de credenciais com Google. Podemos usar o provedor nomeado para Google e modificar nosso código de agente para chamar o provedor de credenciais e usar o access_token para obter os eventos de calendário ou agenda do usuário do Google.

### Arquitetura do Tutorial

<div style="text-align:center">
    <img src="images/outbound_auth_3lo.png" width="90%"/>
</div>


### Detalhes do Tutorial

| Informação          | Detalhes                                                                      |
|:--------------------|:------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional                                                                |
| Tipo de agente      | Único                                                                         |
| Framework Agêntico  | Strands Agents                                                                |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                    |
| Componentes         | Hospedagem de agente no AgentCore Runtime. Usando Strands Agent e Claude Model |
| Vertical            | Cross-vertical                                                                |
| Complexidade        | Média                                                                         |
| SDK usado           | Amazon BedrockAgentCore Python SDK e boto3                                    |
| Provedor Credential | Tipo : OAuth2 - Google Provider                                               |


### Funcionalidades Chave do Tutorial

* Hospedagem de Agentes no Amazon Bedrock AgentCore Runtime
* Uso de modelos Claude
* Uso de Strands Agents
* Uso de AgentCore egress Auth com provedor de credenciais OAuth2 Google.

## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker em execução

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Configurar Inbound Auth com Cognito como IDP
Vamos provisionar um Userpool Cognito com um App client e um usuário de teste. Usaremos Amazon Cognito para fornecer tokens JWT para acessar nosso servidor MCP implantado. Para fazer isso, usaremos a função de suporte `setup_cognito_user_pool` do nosso script `utils`.

Nota: O access_token do Cognito é válido apenas por 2 horas. Se o access_token expirar, você pode gerar outro access_token usando o método `reauthenticate_user`.

In [ ]:
import sys
import os

# Get the current notebook's directory
current_dir = os.path.dirname(
    os.path.abspath("__file__" if "__file__" in globals() else ".")
)

utils_dir = os.path.join(current_dir, "..")
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

In [ ]:
import subprocess
from boto3.session import Session
from utils import setup_cognito_user_pool, reauthenticate_user

boto_session = Session()
region = boto_session.region_name

print(f"Region: {region}")

identity_client = boto_session.client("bedrock-agentcore-control")

In [ ]:
region

In [ ]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool("Cognito_3LO_Google")
print("Cognito setup completed ✓")

## Configurar Google para OAuth2 (Fluxo em nome do usuário)
Siga estes passos para registrar sua aplicação, criar um projeto e configurar credenciais OAuth para acesso ao Google Calendar:

Nesta seção, configuraremos seu Google para acessar a API do Google Calendar com permissões somente leitura.

1. Criar um Projeto no Google Developer Console
    1.    Acesse o [Google Developer Console](https://console.developers.google.com/)
    2.    Na barra de navegação superior, clique em "Create Project"
    3.    Insira um Nome de Projeto
    4.    Escolha uma Organização ou deixe como "No organization" se não aplicável
    5.    Clique em Create. Seu novo projeto aparecerá na lista de projetos.
2. Habilitar Google Calendar API
    1.    Com seu projeto selecionado (usando o checkbox), abra o menu lateral (menu hambúrguer) e vá para APIs & Services > Library
    2.    Na barra de pesquisa, digite Google Calendar API
    3.    Clique em Google Calendar API nos resultados, então clique em Enable
3. Configurar OAuth Consent Screen
    1.    No menu lateral, vá para APIs & Services > OAuth consent screen
    2.    Clique em "Get started"
    3.    Preencha os campos obrigatórios:
    4.    App Name
    5.    User Support Email
    6. Clique em Next e então selecione o tipo de Audience correto, i.e. Internal ou External, Clique em Next (Se selecionar External, garanta que inseriu email ids de usuários aqui, para poder testar com esses usuários)
    7.    Developer Contact Information, Insira seu email id
    8.  Clique em "Finish" após aceitar os termos e condições e então clique em "Create"
4. Adicionar seu email id do Google como usuário de teste
    1.    No menu lateral, vá para APIs & Services > OAuth consent screen
    2.    Selecione "Audience" no menu do lado esquerdo
    3.    Em Test users, clique em "+ Add Users" e adicione seu gmail id para a conta Google
5. Criar OAuth 2.0 Credentials
    1.    Vá para APIs & Services no menu lateral e então selecione > Credentials
    2.    Clique em Create Credentials > OAuth client ID
    3.    Selecione Web application como o tipo de aplicação
    4.    Insira um nome para as credenciais
    5.    Clique em Create
6. Obter Client ID e Client Secret
    1.    Após criação, uma janela exibirá seu Client ID e Client Secret. Copie-os para uso posterior
    2.    Baixe as credenciais como arquivo JSON ou copie-as para uso na configuração da sua aplicação
7. Atualizar o Data access
    1. Vá para APIs & Services no menu lateral e então selecione > Credentials
    2. Selecione a "web app" que você criou no passo 4d
    3. Selecione "Data access" no menu lateral
    4. Clique em "Add or remove scopes"
    5. Adicione escopos baseados no seu caso de uso. Ex: Para Google calendar adicione, "https://www.googleapis.com/auth/calendar.readonly" em "Manually add scopes" e clique update seguido de Save/Update
    6. Clique em "Save" novamente na página "Data access" para salvar sua configuração
8. Usar as Credenciais no Seu Agente
    1.    Na próxima seção, configuraremos um provedor de credenciais de recurso para usar o Client ID, Client Secret e Redirect URI para o fluxo OAuth 3-legged.

## OAuth2 Authorization URL Session Binding Process

The OAuth2 authorization URL session binding process is a critical security mechanism that ensures OAuth2 authorization sessions are properly associated with authenticated users in AgentCore Identity. This process prevents session hijacking and ensures that OAuth tokens are only granted to the intended user.

Ref : https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html

### How session binding works
<div style="text-align:center">
    <img src="images/identity-session-binding.png" width="90%"/>
</div>

1. Invoke agent – Your agent code invokes GetResourceOauth2Token API to retrieve an authorization URL, when an originating agent user wants to access some application or resource that he/she owns.

2. Generate authorization URL – AgentCore Identity generates an authorization URL and session URI for the user to navigate to and consent access.

3. Authorize and obtain access token – The user navigates to the authorization URL and grants consent for your agent to access his/her resource. After that, AgentCore Identity redirects the user's browser to your HTTPS application endpoint with information containing the originating user of the authorization request. At this point, your HTTPS application endpoint determines if the originating agent user is still the same as the currently logged in user of your application. If they match, your application endpoint invokes CompleteResourceTokenAuth so that AgentCore Identity can fetch and store the access token.

4. Re-invoke agent to obtain access token – Once the application returns a valid response, your agent application will be able to retrieve the OAuth2.0 access tokens that were originally requested for the user. If the users do not match, your application simply does nothing or logs the attempt.

By allowing your application endpoint to verify the user identity, AgentCore Identity allows your agent application to ensure that it is always the same user who initiated the authorization request and the one who consented access.

### Overview of the OAuth2 Session Binding Flow in this sample

The OAuth2 session binding process involves several key steps that coordinate between your application, AgentCore Identity, external OAuth providers (like Google, Github ), and a local callback server:

#### Step 1: Create Application Callback URL
- Create a publicly accessible HTTPS callback endpoint in your application
- This endpoint will handle OAuth redirects and validate user sessions
- Example: `https://myagentapp.com/callback`

#### Step 2: Update Workload Identity with Callback URL
- Register your callback URL as an `AllowedResourceOauth2ReturnUrl` in the workload identity
- This is accomplished using the `UpdateWorkloadIdentity` API
- **In this tutorial**: This step is handled automatically by the code below that updates the workload identity with the local callback server URL

#### Step 3: Create OAuth2 Credential Provider
- Configure the credential provider with external OAuth provider details (client ID, secret)
- AgentCore Identity returns a unique callback URL for the provider
- Register this callback URL with the external OAuth provider (e.g., Google Console)

#### Step 4: Implement Session Validation and Token Completion
- Your callback endpoint must validate the current user's session
- Call `CompleteResourceTokenAuth` API with the user identifier and session URI
- **In this tutorial**: The `oauth2_callback_server.py` handles this automatically

#### Step 5: OAuth Flow Execution
- User triggers OAuth flow through agent interaction
- User is redirected to external provider for authorization
- Provider redirects back to your callback with session information
- Session binding completes and OAuth tokens become available

### Local Development with oauth2_callback_server.py

For local development and testing, this tutorial uses `oauth2_callback_server.py` which:

1. **Runs a Local FastAPI Server** (`localhost:9090`)
   - Provides `/ping` endpoint for health checks
   - Provides `/userIdentifier/token` endpoint to store user tokens
   - Provides `/oauth2/callback` endpoint to handle OAuth redirects

2. **Manages User Token Storage**
   - Stores the user's JWT token from Cognito authentication
   - Associates OAuth sessions with the correct user identity

3. **Handles OAuth Callback Processing**
   - Receives OAuth redirects with `session_id` parameter
   - Calls `CompleteResourceTokenAuth` to bind the session
   - Validates user identity before completing the flow

4. **Provides Session Security**
   - Ensures OAuth sessions are bound to authenticated users
   - Prevents unauthorized access to OAuth tokens

### Integration with Workload Identity Update

The code snippet you referenced performs a crucial step in the OAuth2 session binding process:

```python
if launch_result.agent_id:
    workload_name = launch_result.agent_id
    workload_identity = identity_client.get_workload_identity(name=workload_name)
    allowed_resource_oauth_2_return_urls = workload_identity.get("allowedResourceOauth2ReturnUrls") or []
    oauth2_callback_url = get_oauth2_callback_url()
    print(f"Updating workload {workload_name} with callback url {oauth2_callback_url}")

    updated_workload_identity = identity_client.update_workload_identity(
        name=workload_name,
        allowed_resource_oauth_2_return_urls=[*allowed_resource_oauth_2_return_urls, oauth2_callback_url],
    )
```

This code:
1. **Retrieves the Agent's Workload Identity**: Uses the agent ID from the runtime deployment
2. **Gets Current Allowed URLs**: Fetches existing `allowedResourceOauth2ReturnUrls`
3. **Adds Local Callback URL**: Includes `http://localhost:9090/oauth2/callback` as an allowed return URL
4. **Updates Workload Identity**: Registers the callback URL with AgentCore Identity

This registration is essential because AgentCore Identity will only redirect OAuth callbacks to pre-registered URLs, providing an additional security layer.

### Security Considerations

The OAuth2 session binding process includes several security measures:
- **URL Validation**: Only pre-registered callback URLs are allowed
- **Session Verification**: User sessions must be validated before token completion
- **User Identity Binding**: OAuth sessions are explicitly bound to authenticated users
- **Token Isolation**: Each user's OAuth tokens are isolated and secure

This comprehensive approach ensures that OAuth2 flows are secure and properly attributed to the correct users in multi-user environments.

---

### Configure the Google OAuth2 credential provider.

Create a `.env` file in the current folder copy the following text in it:
```sh
GOOGLE_CLIENT_ID="" #"client id" recorded from Step 6.1 above
GOOGLE_CLIENT_SECRET="" #"client secret" recorded from Step 6.2 above.
```

Once the client id and client secret are updated, run the below code to create a credentials provider for Google. <br>
Resource credential providers in AgentCore Identity act as intelligent intermediaries that manage the complex relationships between agents, identity providers, and resource servers. Each provider encapsulates the specific endpoint configuration required for a particular service or identity system. The service provides built-in providers for popular services including Google, GitHub, Slack, and Salesforce, with authorization server endpoints and provider-specific parameters pre-configured to reduce development effort. AgentCore Identity supports custom configurations through configurable OAuth2 credential providers that can be tailored to work with any OAuth2-compatible resource server.


In [ ]:
%%writefile .env
GOOGLE_CLIENT_ID="" #"client id" recorded from Step 6.1 above
GOOGLE_CLIENT_SECRET="" #"client secret" recorded from Step 6.2 above.

In [ ]:
import dotenv

dotenv.load_dotenv(override=True)

# Configure Google OAuth2 provider - On-Behalf-Of User
google_provider = identity_client.create_oauth2_credential_provider(
    **{
        "name": "google-cal-provider",
        "credentialProviderVendor": "GoogleOauth2",
        "oauth2ProviderConfigInput": {
            "googleOauth2ProviderConfig": {
                "clientId": os.environ["GOOGLE_CLIENT_ID"],
                "clientSecret": os.environ["GOOGLE_CLIENT_SECRET"],
            }
        },
    }
)
print(google_provider)
print("\n")
print(f"callbackUrl: {google_provider['callbackUrl']}")

## Atualizar a callback url no Google/OAuth 2.0 client
Navegue de volta para o [Google Developer Console](https://console.developers.google.com/)

1. Selecione o Projeto no Google Developer Console
    1.    Selecione o projeto criado anteriormente
2. Atualizar a callback uri
    1.    Vá para APIs & Services no menu lateral e então selecione > Credentials
    2.    Clique no client em "OAuth 2.0 Client IDs"
    3.    Em "Authorised redirect URIs", insira a callback url da etapa anterior. A callback url foi impressa para que você possa facilmente copiá-la da etapa anterior
    4.    Clique em Save

## Preparando seu agente para implantação no AgentCore Runtime

### Strands Agent com um modelo hospedado no Amazon Bedrock
Aqui está um código de agente Strands que inclui o seguinte:
1. Cria uma nova ferramenta chamada "get_calendar_events_today", para obter os eventos do seu Google Calendar para hoje
2. Usa o Provedor de Credenciais criado na etapa anterior para buscar o access_token do Google. Isso inclui o fluxo de consentimento do usuário onde o consentimento é enviado ao usuário para aprovação como parte do fluxo 3LO
3. O agente Strands chama a ferramenta para quaisquer requisições de usuário relacionadas à agenda do usuário.

In [ ]:
# Get the OAuth2 callback URL based on the current environment (notebook/SageMaker)
# This is evaluated HERE in the notebook, not in the agent container
from oauth2_callback_server import get_oauth2_callback_url

oauth2_callback_url_for_agent = get_oauth2_callback_url()

print(
    f"Callback URL for agent (determined from notebook environment): {oauth2_callback_url_for_agent}"
)

## Implantando o agente no AgentCore Runtime
A operação CreateAgentRuntime suporta opções abrangentes de configuração, permitindo que você especifique imagens de container, variáveis de ambiente e configurações de criptografia. Você também pode configurar definições de protocolo (HTTP, MCP) e mecanismos de autorização para controlar como seus clientes se comunicam com o agente.

Nota: A melhor prática de operações é empacotar código como container e enviar para ECR usando pipelines CI/CD e IaC

Neste tutorial, usaremos o Amazon Bedrock AgentCore Python SDK para facilmente empacotar seus artefatos e implantá-los no AgentCore runtime.

### Configurar implantação AgentCore Runtime

Em seguida, usaremos nosso starter toolkit para configurar a implantação do AgentCore Runtime com um entrypoint, a execution role que acabamos de criar e um arquivo de requirements. Também configuraremos o starter kit para criar automaticamente o repositório Amazon ECR na inicialização.

Durante a etapa de configuração, seu docker file será gerado com base no código da sua aplicação

Nota: O authorizer_configuration está configurado para Inbound Auth com Cognito.

<div style="text-align:left">
    <img src="images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

print(f"Region: {region}")

discovery_url = cognito_config.get("discovery_url", "")
client_id = cognito_config.get("client_id", "")
agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_claude_google_3lo.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    memory_mode="NO_MEMORY",
    agent_name="strands_agent_google_3lo",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id],
        }
    },
)
print(response)

## Revisar a configuração do AgentCore

In [ ]:
!cat .bedrock_agentcore.yaml

### Iniciando agente no AgentCore Runtime

Agora que temos um docker file, vamos iniciar o agente no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
from oauth2_callback_server import get_oauth2_callback_url

# Deploy the agent to AgentCore Runtime and get deployment details
launch_result = agentcore_runtime.launch(
    env_vars={"CALLBACK_URL": oauth2_callback_url_for_agent},
    auto_update_on_conflict=True,
)
print(launch_result)

if launch_result.agent_id:
    # Extract the workload name from the deployed agent's ID for identity management
    workload_name = launch_result.agent_id
    # Retrieve the current workload identity configuration from AgentCore Identity
    workload_identity = identity_client.get_workload_identity(name=workload_name)
    # Extract existing OAuth2 callback URLs that are already registered for this workload
    allowed_resource_oauth_2_return_urls = (
        workload_identity.get("allowedResourceOauth2ReturnUrls") or []
    )
    # Get the local OAuth2 callback server URL for session binding (localhost:9090/oauth2/callback)
    oauth2_callback_url = get_oauth2_callback_url()
    print(f"Updating workload {workload_name} with callback url {oauth2_callback_url}")

    # Register the local callback URL with the workload identity to enable OAuth2 session binding
    updated_workload_identity = identity_client.update_workload_identity(
        name=workload_name,
        allowedResourceOauth2ReturnUrls=[
            *allowed_resource_oauth_2_return_urls,
            oauth2_callback_url,
        ],
    )
    print(updated_workload_identity)

#### Adicionar políticas extras necessárias à role auto-criada

Se você está usando isto em uma nova conta onde o modelo não foi acessado antes, você precisará adicionar políticas extras necessárias à role auto-criada para permitir que o agente acesse o modelo.

In [ ]:
import json
import boto3

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

runtime_response = agentcore_control_client.get_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)
runtime_role = runtime_response["roleArn"]
account = boto_session.client("sts").get_caller_identity().get("Account")

policies_to_add = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockModelAccess",
            "Effect": "Allow",
            "Action": [
                "aws-marketplace:ViewSubscriptions",
                "aws-marketplace:Subscribe",
            ],
            "Resource": "*",
        },
        {
            "Sid": "Oauth2TokenAccess",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetResourceOauth2Token",
            ],
            "Resource": "*",
        },
        {
            "Sid": "SecretsManagerAccess",
            "Effect": "Allow",
            "Action": [
                "secretsmanager:GetSecretValue",
            ],
            "Resource": [
                f"arn:aws:secretsmanager:{region}:{account}:secret:secret:bedrock-agentcore-identity!default/oauth2/google-cal-provider*"
            ],
        },
    ],
}
iam_client = boto3.client("iam", region_name=region)

response = iam_client.put_role_policy(
    PolicyDocument=json.dumps(policies_to_add),
    PolicyName="outbound_policies",
    RoleName=runtime_role.split("/")[1],
)

### Verificando o Status do AgentCore Runtime
Agora que implantamos o AgentCore Runtime, vamos verificar seu status de implantação

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
print(f"Final status: {status}")

### Invocando AgentCore Runtime

Finalmente, podemos invocar nosso AgentCore Runtime com um payload

Você notará que o agente chama a ferramenta "Get_calendar_events_today" e dispara o fluxo OAuth 3 Legged. Você será apresentado com a URL de Autorização. Clique na URL de Autorização OU copie/cole ela em uma nova sessão/aba do navegador para completar o fluxo de consentimento do usuário.
Uma vez que a Autorização completar, O provedor de credenciais "google-cal-provider" buscará o access_token do Google e completará a execução da ferramenta para buscar os eventos do seu calendário.

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
from oauth2_callback_server import (
    store_token_in_oauth2_callback_server,
    wait_for_oauth2_server_to_be_ready,
)

bearer_token = reauthenticate_user(cognito_config.get("client_id"))

oauth2_callback_server_cmd = [
    sys.executable,
    "oauth2_callback_server.py",
    "--region",
    region,
]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

try:
    # Start the OAuth2 callback server
    successfully_started_oauth2_server = wait_for_oauth2_server_to_be_ready()
    if not successfully_started_oauth2_server:
        print(
            "Failed to start OAuth2 callback server to handle session binding "
            "(https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)"
        )
    else:
        store_token_in_oauth2_callback_server(bearer_token)
        invoke_response = agentcore_runtime.invoke(
            {"prompt": "What is in my agenda for today? Highlight the main events!"},
            bearer_token=bearer_token,
        )
        print(invoke_response)
finally:
    oauth2_callback_server_process.terminate()

## Opcional - Testar o agente usando uma Aplicação Streamlit

Você pode testar seu agente implantado usando uma aplicação web Streamlit que fornece uma interface de chat agradável. O arquivo `chatbot_app_cognito.py` neste diretório cria um chatbot baseado na web que:

- Lê automaticamente a configuração de `.bedrock_agentcore.yaml`
- Fornece autenticação Cognito
- Mostra uma interface de chat moderna com respostas em streaming
- Manipula o fluxo OAuth 3LO para acesso ao Google Calendar

### Executando a Aplicação Streamlit

Você pode executar a aplicação Streamlit de várias maneiras:

#### Opção 1: Executar do Jupyter Notebook (Diretório Atual)
- Execute a célula abaixo para iniciar a aplicação Streamlit diretamente deste notebook:
- Login: Use as credenciais testuser / MyPassword123! (o usuário de teste padrão criado pela configuração Cognito)
- Teste alguns prompts simples como "Conte-me uma piada"
- Teste com um prompt que disparará a ferramenta `Get_calendar_events_today` como "O que está na minha agenda para hoje?"
- Você verá a URL de Autorização retornada. Clique na url ou copie/cole a url para uma nova aba/janela do navegador para completar o fluxo de consentimento do usuário.

In [ ]:
from chatbot_app_cognito import get_streamlit_url

# Change to the current directory where the chatbot_app_cognito.py file is located
notebook_dir = os.getcwd()

# Start the Streamlit app
print("Starting Streamlit app...")

oauth2_callback_server_cmd = [
    sys.executable,
    "oauth2_callback_server.py",
    "--region",
    region,
]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

try:
    wait_for_oauth2_server_to_be_ready()

    # Run streamlit in the current directory
    process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "chatbot_app_cognito.py",
            "--server.port=8501",
            "--server.showEmailPrompt=false",
        ],
        cwd=notebook_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    # Print the output as it comes
    for line in iter(process.stdout.readline, ""):
        if line:
            if "8501" in line:
                print("\n🎉 Streamlit app is ready!")
                streamlit_url = get_streamlit_url()
                print(f"\n🚀 Streamlit Application URL:\n{streamlit_url}\n")
                print(
                    "⚠️ To stop the app, interrupt the kernel or press Ctrl+C in the terminal"
                )
                break

except KeyboardInterrupt:
    print("\nStreamlit app stopped.")
    oauth2_callback_server_process.terminate()
    process.terminate()
except Exception as e:
    print(f"Error starting Streamlit app: {e}")

#### Opção 2: Executar do Terminal

Alternativamente, você pode executar a aplicação Streamlit do seu terminal:

```bash
# Navegue para o diretório atual
cd 01-tutorials/03-AgentCore-identity/05-Outbound_Auth_3lo/

# Execute a aplicação Streamlit
python oauth2_callback_server.py -r <region> & streamlit run chatbot_app_cognito.py
```

#### Usando a Aplicação Streamlit

1. **Login**: Use as credenciais `testuser` / `MyPassword123!` (o usuário de teste padrão criado pela configuração Cognito)
2. **Chat**: Faça perguntas como "O que está na minha agenda para hoje?" ou "Destaque eventos principais da minha agenda de hoje"
3. **Fluxo OAuth**: Quando o agente precisar de acesso ao Google Calendar, você verá uma URL de autorização - clique nela para completar o fluxo OAuth
4. **Funcionalidades**: A aplicação inclui:
   - Respostas em streaming em tempo real
   - URLs clicáveis
   - Interface de chat moderna
   - Consciência de contexto
   - Tratamento de erros com mensagens informativas

A aplicação lê automaticamente toda a configuração do seu arquivo `.bedrock_agentcore.yaml`, então ela usará o mesmo agent runtime que você acabou de implantar.

## Limpeza (Opcional)

- Vamos agora limpar o AgentCore Runtime criado
- Descomente as Células abaixo e execute.

In [ ]:
# launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
# agentcore_control_client = boto3.client(
#     'bedrock-agentcore-control',
#     region_name=region
# )
# ecr_client = boto3.client(
#     'ecr',
#     region_name=region

# )

# runtime_delete_response = agentcore_control_client.delete_agent_runtime(
#     agentRuntimeId=launch_result.agent_id,

# )

# response = ecr_client.delete_repository(
#     repositoryName=launch_result.ecr_uri.split('/')[1],
#     force=True
# )

## Parabéns!